In [3]:
path = '/kaggle/input/finaldata123/normal_labeled.log'

In [2]:
import re
from typing import Union, Iterable, List, Dict, Optional
import pandas as pd
NGINX_REGEX_STRICT = re.compile(
    r'(?P<ip>\S+)\s+-\s+-\s+'                              # IP - -
    r'\[(?P<timestamp>[^\]]+)]\s+'                         # [timestamp]
    r'"(?P<method>[A-Z]+)\s+'                              # "METHOD␣
    r'(?P<url>.+?)\s+'                                     # URL (non-greedy)
    r'(?P<protocol>[A-Z]+/\d(?:\.\d)?)"\s+'                # PROTOCOL"
    r'(?P<status>\d{3}|-)\s+'                              # status
    r'(?P<size>\d+|-)\s+'                                  # size
    r'"(?P<referrer>[^"]*)"\s+'                            # "referrer"
    r'"(?P<user_agent>[^"]*)"'                            # "user-agent"
    r'(?:[ \t]+(?P<label>[01]))?$',        #  ← thêm nhóm label tuỳ chọn
    flags=re.IGNORECASE,
)

NGINX_REGEX_FALLBACK = re.compile(
    r'(?P<ip>\S+)\s+-\s+-\s+'                              # IP - -
    r'\[(?P<timestamp>[^\]]+)]\s+'                         # [timestamp]
    r'(?P<method>[A-Z]+)\s+'                               # METHOD
    r'(?P<url>.+?)\s+'                                     # URL
    r'(?P<protocol>[A-Z]+/\d(?:\.\d)?)\s+'                 # PROTOCOL
    r'(?P<status>\d{3}|-)\s+'                              # status
    r'(?P<size>\d+|-)\s+'                                  # size
    r'(?P<referrer>\S+|-)\s+'                              # referrer (không quotes)
    r'(?P<user_agent>.+)'                                 # user-agent (còn lại)
    r'(?:[ \t]+(?P<label>[01]))?$',        #  ← thêm nhóm label tuỳ chọn

    flags=re.IGNORECASE,
)

# Gộp thành tuple để lần lượt thử
NGINX_COMBINED_PATTERNS = (NGINX_REGEX_STRICT, NGINX_REGEX_FALLBACK)

# ──────────────────────────────────────────────────────────────
# 2) Tiện ích: loại bỏ ký tự control (nếu log bị lẫn \x00 …)
# ──────────────────────────────────────────────────────────────
def strip_control(s: str) -> str:
    """Remove leading control chars (0x00–0x1F) ở đầu dòng."""
    return re.sub(r'^[\x00-\x1F]+', "", s)

# ──────────────────────────────────────────────────────────────
# 3) Hàm wrapper parse_nginx_log
# ──────────────────────────────────────────────────────────────
def parse_nginx_log(
    source: Union[str, Iterable[str]],
    patterns: Iterable[re.Pattern] = NGINX_COMBINED_PATTERNS,
    as_dataframe: bool = True,
    encoding: Optional[str] = "utf-8",
) -> Union[pd.DataFrame, List[Dict[str, str]]]:
    """
    Parse log Nginx / Apache (combined) thành list[dict] hoặc pandas.DataFrame.

    Args:
        source (str | Iterable[str]):
            • Chuỗi đường dẫn file, hoặc
            • Iterable (list, generator, ...) các dòng log.
        patterns (Iterable[re.Pattern]): Danh sách regex sẽ thử lần lượt.
        as_dataframe (bool): True -> trả về DataFrame, False -> list[dict].
        encoding (str | None): Encoding khi mở file (nếu source là path).

    Returns:
        pandas.DataFrame | list[dict]
    """
    # 1) Lấy iterator dòng log
    if isinstance(source, str):                # truyền path
        fh = open(source, "r", encoding=encoding, errors="replace")
        lines = fh
        close_file = True
    else:                                      # iterable dòng
        lines = source
        close_file = False

    # 2) Parse
    parsed: List[Dict[str, str]] = []
    for raw_line in lines:
        line = strip_control(raw_line.rstrip("\n"))
        for pat in patterns:
            m = pat.match(line)
            if m:
                parsed.append(m.groupdict())
                break                          # matched → sang dòng kế
        # nếu muốn ghi lại MISS, thêm else: missed.append(line)

    # 3) Đóng file nếu cần
    if close_file:
        fh.close()

    # 4) Trả kết quả
    return pd.DataFrame(parsed) if as_dataframe else parsed

In [4]:
df = parse_nginx_log(path)


In [7]:
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)

In [8]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df['label'] # Rất quan trọng để giữ tỷ lệ label trong cả 2 tập
)

print("Kích thước tập Train:", train_df.shape)
print("Kích thước tập Test:", test_df.shape)

Kích thước tập Train: (83946, 10)
Kích thước tập Test: (20987, 10)


In [9]:
df.isna().sum()/len(df)

ip            0.0
timestamp     0.0
method        0.0
url           0.0
protocol      0.0
status        0.0
size          0.0
referrer      0.0
user_agent    0.0
label         0.0
dtype: float64

In [10]:
def timestamp_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Nhận vào một DataFrame chứa cột 'timestamp' và trả về DataFrame
    đã được bổ sung đầy đủ các feature về thời gian.
    """
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], format='%d/%b/%Y:%H:%M:%S %z', errors='coerce')
    
    df = df.sort_values('timestamp_dt').reset_index(drop=True)

    df['hour_of_day'] = df['timestamp_dt'].dt.hour
    df['day_of_week'] = df['timestamp_dt'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

    def get_part_of_day(hour):
        if 5 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 17:
            return 'afternoon'
        elif 17 <= hour < 21:
            return 'evening'
        else:
            return 'night'
    df['part_of_day'] = df['hour_of_day'].apply(get_part_of_day)

    df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    df['time_since_last_event'] = df['timestamp_dt'].diff().dt.total_seconds().fillna(0)
    df.drop(columns=["timestamp_dt"], inplace=True)
    df.drop(columns=["timestamp"], inplace=True)

    return df

In [11]:
def status_features(df):
    """
    Trích xuất các feature từ cột 'status' và 'size' trong log web.

    Args:
        df (pd.DataFrame): DataFrame chứa ít nhất 2 cột: 'status' (int), 'size' (int)

    Returns:
        pd.DataFrame: DataFrame gốc kèm thêm các cột đặc trưng mới.
    """
    # Kiểm tra cột trước
    if 'status' not in df.columns or 'size' not in df.columns:
        raise ValueError("DataFrame cần có cột 'status' và 'size'.")

    df = df.copy()
    df['status'] = pd.to_numeric(df['status'], errors='coerce').fillna(0).astype(int)
    df['size'] = pd.to_numeric(df['size'], errors='coerce').fillna(0).astype(int)
    # 1. 4xx - lỗi phía client
    df['status_is_client_error'] = df['status'].apply(lambda x: 1 if 400 <= x < 500 else 0)

    # 2. 5xx - lỗi phía server
    df['status_is_server_error'] = df['status'].apply(lambda x: 1 if 500 <= x < 600 else 0)

    # 3. Lỗi nói chung
    df['status_is_error'] = ((df['status_is_client_error'] == 1) | (df['status_is_server_error'] == 1)).astype(int)

    # 4. Thành công (2xx)
    df['status_is_success'] = df['status'].apply(lambda x: 1 if 200 <= x < 300 else 0)

    # 5. Redirect (3xx)
    df['status_is_redirect'] = df['status'].apply(lambda x: 1 if 300 <= x < 400 else 0)

    # 6. Response size bằng 0
    df['size_is_zero'] = df['size'].apply(lambda x: 1 if x == 0 else 0)

    return df

In [12]:
from urllib.parse import urlparse
def calculate_entropy(text_string: str) -> float:
    """
    Tính entropy của chuỗi ký tự (dựa trên xác suất xuất hiện ký tự).
    """
    import math
    from collections import Counter

    if not text_string:
        return 0.0

    counts = Counter(text_string)
    total = len(text_string)
    entropy = -sum((count / total) * math.log2(count / total) for count in counts.values())
    return entropy

def referrer_features(df):
    df = df.copy()
    
    df['referrer_len'] = df['referrer'].astype(str).apply(len)
    df['referrer_entropy'] = df['referrer'].astype(str).apply(calculate_entropy)
    df['referrer_is_empty'] = df['referrer'].apply(lambda x: 1 if x == '-' or pd.isna(x) or str(x).strip() == '' else 0)

    def get_referrer_domain(ref):
        ref_str = str(ref)
        if pd.isna(ref) or ref_str == '-' or not ref_str.startswith('http'):
            return 'none'
        try:
            parsed = urlparse(ref_str)
            return parsed.netloc.lower() if parsed.netloc else 'unknown_format_but_not_empty'
        except:
            return 'parse_error'

    df['referrer_domain'] = df['referrer'].apply(get_referrer_domain)
    df['referrer_is_external_or_valid'] = df['referrer_domain'].apply(
        lambda x: 0 if x in ['none', 'parse_error', 'unknown_format_but_not_empty'] else 1
    )

    return df

In [13]:
cat_col = df.select_dtypes(include=["object", "category"]).columns.tolist()

num_col = df.select_dtypes(include=["number"]).columns.tolist()

In [14]:
print(cat_col)
print(num_col)

['ip', 'timestamp', 'method', 'url', 'protocol', 'status', 'size', 'referrer', 'user_agent']
['label']


In [16]:
import numpy as np
# Cell 7: Áp dụng Feature Engineering
print("Processing Train set...")
train_featured = timestamp_features(train_df)
train_featured = status_features(train_featured)
train_featured = referrer_features(train_featured)

print("Processing Test set...")
test_featured = timestamp_features(test_df)
test_featured = status_features(test_featured)
test_featured = referrer_features(test_featured)

Processing Train set...
Processing Test set...


In [17]:
df.isna().sum()/len(df)

ip            0.0
timestamp     0.0
method        0.0
url           0.0
protocol      0.0
status        0.0
size          0.0
referrer      0.0
user_agent    0.0
label         0.0
dtype: float64

In [ ]:
# Cell 8: Chuẩn bị X, y
TARGET = 'label'
ua_cols_to_drop = [col for col in train_featured.columns if col.startswith('ua_')]

# Dọn dẹp các cột không cần thiết cho model
cols_to_drop = [
    'ip', 'url', 'referrer', 'user_agent', 
    'ua_browser_version_major', 'ua_os_version_major', 'ua_device_brand', 'is_suspicious', 'ua_is_tool', 'status_is_error'
] + ua_cols_to_drop

X_train = train_featured.drop(columns=[TARGET] + [col for col in cols_to_drop if col in train_featured.columns])
y_train = train_featured[TARGET]

X_test = test_featured.drop(columns=[TARGET] + [col for col in cols_to_drop if col in test_featured.columns])
y_test = test_featured[TARGET]

# Đảm bảo các cột trong X_train và X_test khớp nhau
X_test = X_test[X_train.columns]

# Cell 9: Huấn luyện CatBoost (giống như code của bạn)
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report

cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

model = CatBoostClassifier(
    verbose=100, 
    random_state=42,
)


model.fit(train_pool) # Hoặc fit như cũ

# Đánh giá
y_pred = model.predict(test_pool)
print(classification_report(y_test, y_pred))

Learning rate set to 0.06831
0:	learn: 0.4730627	total: 148ms	remaining: 2m 27s
100:	learn: 0.0084641	total: 6.55s	remaining: 58.3s
200:	learn: 0.0074509	total: 12.7s	remaining: 50.3s
300:	learn: 0.0065618	total: 19.2s	remaining: 44.6s
400:	learn: 0.0058373	total: 25.7s	remaining: 38.4s
500:	learn: 0.0052983	total: 32.9s	remaining: 32.7s
600:	learn: 0.0048186	total: 39.6s	remaining: 26.3s
700:	learn: 0.0044293	total: 46.4s	remaining: 19.8s
800:	learn: 0.0041608	total: 52.9s	remaining: 13.1s
900:	learn: 0.0038443	total: 59.4s	remaining: 6.53s
999:	learn: 0.0036709	total: 1m 6s	remaining: 0us
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20000
           1       1.00      0.95      0.97       987

    accuracy                           1.00     20987
   macro avg       1.00      0.98      0.99     20987
weighted avg       1.00      1.00      1.00     20987



This is the anomaly detection problem so the accuracy won'r reflect anything - because in case 99 normal data points, 1 anomaly datapoints, model only predict normal data points will get 99% accuracy -> so the model's performance will be judged on the f1-score & recall, and the 0.95 recall is not called overfitting.